# AgenticML — Experimento 2: Titanic (Classificação Binária)

**Dataset:** Titanic (seaborn)  
**Tipo de problema:** Classificação binária (sobreviveu / não sobreviveu)  
**Objetivo:** Prever se um passageiro sobreviveu ao naufrágio do Titanic  
**Referência de comparação:** Auto-sklearn (seção 9.1 da spec)  

---
**Relevância:** Este experimento é mais rico que o Iris pois exercita:
- Agente 2: detecção de missing values (coluna `age`) e desbalanceamento de classes
- Agente 3: imputação, One-Hot Encoding em variáveis nominais (`sex`, `embarked`, `class`)
- Agente 5: comparação de modelos em dataset com ruído real

> ⚠️ **Modo fallback:** Configure `.env` com chave de API para ativar explicações LLM completas.

In [5]:
import sys, os, time, warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import seaborn as sns

from core.graph import build_graph
from core.state import initial_state

print('Ambiente configurado.')

Ambiente configurado.


## 1. Dataset — Titanic

| Atributo | Valor |
|----------|-------|
| Instâncias | ~891 |
| Features selecionadas | pclass, sex, age, sibsp, parch, fare, embarked |
| Target | survived (0 = não sobreviveu, 1 = sobreviveu) |
| Missing values | age (~20%), embarked (2 linhas) |
| Desbalanceamento | ~62% não sobreviveram / ~38% sobreviveram |
| Fonte | `seaborn.load_dataset('titanic')` |

In [6]:
titanic_raw = sns.load_dataset('titanic')

# Seleciona colunas relevantes (remove duplicatas/leakage)
COLS = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'survived']
df = titanic_raw[COLS].copy()

print('Shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\nDistribuição da target:')
print(df['survived'].value_counts())
print(f'  → Classe majoritária: {df["survived"].value_counts(normalize=True).max()*100:.1f}%')
display(df.head())

Shape: (891, 8)

Missing values:
age         177
embarked      2
dtype: int64

Distribuição da target:
survived
0    549
1    342
Name: count, dtype: int64
  → Classe majoritária: 61.6%


,pclass,sex,age,sibsp,parch,fare,embarked,survived
0,3,male,22.0,1,0,7.2500,S,0
1,1,female,38.0,1,0,71.2833,C,1
2,3,female,26.0,0,0,7.9250,S,1
3,1,female,35.0,1,0,53.1000,S,1
4,3,male,35.0,0,0,8.0500,S,0


In [7]:
csv_bytes = df.to_csv(index=False).encode('utf-8')
print(f'CSV gerado: {len(csv_bytes):,} bytes')

CSV gerado: 23,182 bytes


## 2. Execução do Pipeline AgenticML

In [8]:
PROMPT = (
    "Prever se um passageiro sobreviveu ao naufrágio do Titanic "
    "com base nas características socioeconômicas e demográficas."
)

state = initial_state(
    dataset_bytes=csv_bytes,
    filename='titanic.csv',
    prompt=PROMPT,
    problem_type_hint='classification',
)

t0 = time.time()
result = build_graph().invoke(state)
elapsed = time.time() - t0

print(f'\n✓ Pipeline concluído em {elapsed:.2f}s')
print('Status final:', result.get('status'))


✓ Pipeline concluído em 13.83s
Status final: done


## 3. Resultados por Agente

### Agente 1 — Ingestão e Inspeção

In [9]:
inspection = result.get('inspection', {})
schema = inspection.get('schema', {})

print('Shape detectado:', schema.get('shape'))
print('Encoding:', inspection.get('encoding_detected'))
print()
stats = inspection.get('descriptive_stats', {})
print('Colunas com nulls detectados pelo Agente 1:')
for col, s in stats.items():
    nulls = s.get('nulls', 0)
    if nulls > 0:
        print(f'  {col}: {nulls} nulos')

Shape detectado: [891, 8]
Encoding: utf-8

Colunas com nulls detectados pelo Agente 1:
  age: 177 nulos
  embarked: 2 nulos


### Agente 2 — Diagnóstico de Qualidade

In [10]:
quality = result.get('quality', {})
issues = quality.get('issues', [])

print(f"Overall severity: {quality.get('overall_severity')}")
print(f"Issues detectados: {len(issues)}\n")

for i in issues:
    print(f"[{i['severity'].upper()}] {i['type']}")
    print(f"  → {i['alert']}\n")

Overall severity: medium
Issues detectados: 5

[MEDIUM] missing_values
  → A coluna 'age' possui 177 valores ausentes, o que representa 19.87% do total. Recomenda-se tratar esses valores ausentes, possivelmente utilizando a imputação com a média ou mediana, ou ainda técnicas mais avançadas como KNN imputation, para evitar perda de informação.

[LOW] missing_values
  → A coluna 'embarked' possui 2 valores ausentes, representando 0.22% do total. Embora a quantidade seja pequena, é importante decidir como lidar com esses valores, seja removendo as linhas correspondentes ou imputando o valor mais frequente.

[LOW] class_imbalance
  → A coluna 'parch' apresenta um desequilíbrio de classes, com a classe '0' representando 76.09% dos dados. Isso pode afetar o desempenho de modelos que assumem uma distribuição balanceada. Considere técnicas de balanceamento como oversampling ou undersampling para mitigar esse problema.

[MEDIUM] duplicate_rows
  → Foram detectadas 111 linhas duplicadas, o que r

### Agente 3 — Feature Engineering

In [11]:
features = result.get('features', {})
transforms = features.get('transformations_applied', [])

print(f"Target:   {features.get('target_column')}")
print(f"Features: {features.get('feature_columns')}")
print()
for t in transforms:
    print(f"• {t['type']} em {t['columns']}")
    print(f"  → {t['justification']}")

Target:   survived
Features: ['pclass', 'age', 'sibsp', 'parch', 'fare', 'sex_male', 'embarked_Q', 'embarked_S']

• impute_median em ['age']
  → A coluna 'age' possui valores ausentes e é uma variável numérica contínua. A mediana é uma boa escolha para imputação, pois é robusta a outliers e a distribuição da idade pode não ser simétrica.
• impute_most_frequent em ['embarked']
  → A coluna 'embarked' possui poucos valores ausentes e é categórica. Imputar com o valor mais frequente é uma estratégia comum para lidar com valores ausentes em variáveis categóricas.
• one_hot_encoding em ['sex']
  → A coluna 'sex' é categórica nominal com apenas duas categorias. O one-hot encoding é apropriado para representar essas categorias sem introduzir uma ordem artificial.
• one_hot_encoding em ['embarked']
  → A coluna 'embarked' é categórica nominal com três categorias. O one-hot encoding é adequado para evitar a introdução de uma ordem artificial entre as categorias.
• standard_scaler em ['age', 'si

### Agente 4 — Interpretação de Linguagem Natural

In [12]:
interp = result.get('interpretation', {})
print(f"Tipo:       {interp.get('problem_type')}")
print(f"Confiança:  {interp.get('confidence')}")
print(f"Divergiu:   {interp.get('diverged_from_user')}")
print(f"\nJustificativa: {interp.get('justification')}")

Tipo:       classification
Confiança:  high
Divergiu:   False

Justificativa: O objetivo do usuário é prever se um passageiro sobreviveu ao naufrágio do Titanic, o que é uma tarefa de classificação, pois envolve prever uma categoria ou classe (sobreviveu ou não sobreviveu). A coluna target identificada é 'survived', que é uma variável binária (0 ou 1), indicando se o passageiro sobreviveu. As características do dataset, como 'pclass', 'age', 'sibsp', 'parch', 'fare', 'sex_male', 'embarked_Q', e 'embarked_S', são usadas para prever essa classe. Portanto, o tipo de problema é inequivocamente de classificação.


### Agente 5 — Seleção e Treinamento

In [13]:
training = result.get('training', {})

print(f"Modelo selecionado: {training.get('model_selected')}")
print(f"F1-score (weighted): {training.get('metric_value')}")
print()

candidates = training.get('candidate_results', [])
df_cands = pd.DataFrame(candidates).sort_values('score', ascending=False)
display(df_cands)

print(f"\nJustificativa: {training.get('justification')}")

Modelo selecionado: XGBoost
F1-score (weighted): 0.8254



,model,score
2,XGBoost,0.8254
3,SVM,0.8238
1,Random Forest,0.8218
0,Logistic Regression,0.7923



Justificativa: XGBoost foi selecionado com F1-score ponderado de 0.8254 para o problema de classification. Demais candidatos avaliados: SVM (0.8238), Random Forest (0.8218), Logistic Regression (0.7923).


### Agente 6 — Previsão e Explicação

In [14]:
output = result.get('output', {})
explanation = output.get('explanation', {})
importances = output.get('feature_importances', [])

print('=== Explicação Técnica ===')
print(explanation.get('technical', ''))

print('\n=== Explicação Acessível ===')
print(explanation.get('accessible', ''))

if importances:
    print('\n=== Importância das Features ===')
    display(pd.DataFrame(importances))

=== Explicação Técnica ===
O pipeline AgenticML foi aplicado a um conjunto de dados com 891 linhas e 8 colunas, focando na previsão de sobrevivência de passageiros do Titanic. O Agente 2 identificou problemas de qualidade, incluindo valores ausentes, desequilíbrio de classes, linhas duplicadas e viés, com severidade geral média. Para mitigar esses problemas, o Agente 3 realizou transformações de engenharia de features, como imputação da mediana para a coluna 'age', imputação do valor mais frequente para 'embarked', e codificação one-hot para 'sex' e 'embarked'. Além disso, aplicou-se a normalização com Standard Scaler em várias colunas numéricas.

O Agente 4 confirmou que o problema é de classificação, com a variável alvo 'survived' sendo binária. O Agente 5 selecionou o modelo XGBoost, que apresentou o melhor desempenho com um F1-score ponderado de 0.8254, superando outros modelos como SVM (0.8238), Random Forest (0.8218) e Logistic Regression (0.7923). As features mais importantes pa

,feature,importance
0,sex_male,0.576737
1,pclass,0.209140
2,sibsp,0.063253
3,fare,0.036787
4,age,0.035149
5,parch,0.029991
6,embarked_S,0.029505
7,embarked_Q,0.019439


## 4. Resumo — Tabela para TCC

In [15]:
schema = result.get('inspection', {}).get('schema', {})
issues = result.get('quality', {}).get('issues', [])
transforms = result.get('features', {}).get('transformations_applied', [])

summary = {
    'Experimento':        'Titanic — Classificação Binária',
    'Dataset':            f"{schema.get('shape', [0,0])[0]} linhas × {schema.get('shape', [0,0])[1]} colunas",
    'Tipo de problema':   interp.get('problem_type', 'N/A'),
    'Issues de qualidade': len(issues),
    'Transformações':     len(transforms),
    'Modelo selecionado': training.get('model_selected', 'N/A'),
    'Métrica':            'f1_weighted',
    'AgenticML score':    training.get('metric_value', 'N/A'),
    'Auto-sklearn score': 'Pendente (Linux/WSL)',
    'Tempo (s)':          round(elapsed, 2),
}

display(pd.DataFrame([summary]).T.rename(columns={0: 'Valor'}))

,Valor
Experimento,Titanic — Classificação Binária
Dataset,891 linhas × 8 colunas
Tipo de problema,classification
Issues de qualidade,5
Transformações,5
Modelo selecionado,XGBoost
Métrica,f1_weighted
AgenticML score,0.8254
Auto-sklearn score,Pendente (Linux/WSL)
Tempo (s),13.83
